In [1]:
## Vloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

In [2]:
import sys
import os

# Acesso aos módulos do diretório
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: C:\pod\hackathon_pod_2025


##### Carregando pacotes

In [3]:
# Pacotes de manipulacao

import pandas as pd
import numpy as np
import os

# Pacotes de visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

# Funcoes customizadas
import configs.function_basic as funcoes

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 1.5.3
numpy: 1.26.4


## Carregando databases

#### Book_03

In [4]:
# Carregando book_03
book_03 = pd.read_parquet(project_root/'database/processed/book_variaveis_03.parquet')
print("Book 03 data shape:", book_03.shape)

Book 03 data shape: (1280828, 101)


In [5]:
book_03.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1280828 entries, 0 to 1280827
Columns: 101 entries, SAFRA to var_93
dtypes: Int64(10), datetime64[ns](2), float64(60), int64(16), object(13)
memory usage: 999.2+ MB


#### Base Recarga

In [6]:
## Carregando todos arquivos em parquet de uma pasta

all_files = [os.path.join(project_root/'database/raw/bases_recarga/bases_recarga/', f) for f in os.listdir(project_root/'database/raw/bases_recarga/bases_recarga/') if f.endswith('.parquet')]
df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]

df_dados_recarga = pd.concat(df_list, ignore_index=True)
print('Base Dados Recarga data shape:', df_dados_recarga.shape)

Base Dados Recarga data shape: (51684470, 24)


In [7]:
df_dados_recarga.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51684470 entries, 0 to 51684469
Data columns (total 24 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   NUM_CPF                object
 1   DW_NUM_NTC             object
 2   DAT_INSERCAO_CREDITO   object
 3   HOR_INSERCAO_CREDITO   object
 4   DW_NUM_CLIENTE         object
 5   COD_TECNOLOGIA_DW      object
 6   COD_CANAL_AQUISICAO    object
 7   COD_TIPO_CREDITO       object
 8   COD_PROMOCAO           object
 9   VAL_CREDITO_INSERIDO   object
 10  VAL_BONUS              object
 11  VAL_REAL               object
 12  COD_PLATAFORMA_ATU     object
 13  COD_STATUS_PLATAFORMA  object
 14  IND_METODO_PAGAMENTO   object
 15  DW_PLANO_TARIFACAO     object
 16  DW_TIPO_RECARGA        object
 17  DW_TIPO_INSERCAO       object
 18  DW_FORMA_PAGAMENTO     object
 19  DW_INSTITUICAO         object
 20  COD_GRUPO_CARTAO       object
 21  DSC_GRUPO_CARTAO_WPP   object
 22  FLAG_SOS               object
 23  VALOR

#### Ajuste no Dataset de Recargas

Para diminuir o tamanho da base processada e focarmos no nosso problema de negócio, iremos filtrar apenas os CPFs que estão na base do `book_03`

Também iremos criar a coluna `SAFRA` a partir da coluna `DAT_INSERCAO_CREDITO`

In [8]:
selecao_publico = book_03['NUM_CPF'].drop_duplicates()

# Selecionando na base de recarga apenas os CPFs presentes na base de score bureau movel
df_base_recarga_selecionada = df_dados_recarga[df_dados_recarga['NUM_CPF'].isin(selecao_publico)]
df_base_recarga_selecionada.head()

,NUM_CPF,DW_NUM_NTC,DAT_INSERCAO_CREDITO,HOR_INSERCAO_CREDITO,DW_NUM_CLIENTE,COD_TECNOLOGIA_DW,COD_CANAL_AQUISICAO,COD_TIPO_CREDITO,COD_PROMOCAO,VAL_CREDITO_INSERIDO,...,IND_METODO_PAGAMENTO,DW_PLANO_TARIFACAO,DW_TIPO_RECARGA,DW_TIPO_INSERCAO,DW_FORMA_PAGAMENTO,DW_INSTITUICAO,COD_GRUPO_CARTAO,DSC_GRUPO_CARTAO_WPP,FLAG_SOS,VALOR_SOS
0,89UU788W7TW,712481754,09OCT2023:00:00:00,170410,1369129520,GSM,17598,PE,-1,20.00,...,A,606234,-2,-2,-2,-2,UB,Rec.Online,0,None
1,W7T8UNX9878,739450945,12MAR2025:00:00:00,32753,1486001611,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,WT,NaoSeAplica,0,None
2,7ZTYY8W8XNY,780682139,27JAN2025:00:00:00,5813,1473175026,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,IW,NaoSeAplica,0,None
3,T97TZT9NUZU,638382020,18DEC2024:00:00:00,3354,1473618173,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,IW,NaoSeAplica,0,None
4,877W7ZX9ZU9,674107712,17DEC2024:00:00:00,21751,1469218303,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,WT,NaoSeAplica,0,None


In [9]:
# Coonferindo se temos valores nulos em DAT_INSERCAO_CREDITO
df_base_recarga_selecionada ['DAT_INSERCAO_CREDITO'].isnull().sum()

0

In [10]:
# Criando coluna SAFRA
df_base_recarga_selecionada = funcoes.criar_coluna_safra(df_base_recarga_selecionada, 'DAT_INSERCAO_CREDITO')

In [11]:
df_base_recarga_selecionada['SAFRA'].value_counts(dropna=False)

202411    3652353
202410    3605527
202412    3584100
202501    3171804
202408    3091771
202409    3080054
202502    2921412
202407    2885957
202503    2769663
202405    2685113
202406    2675063
202403    2638723
202404    2570062
202312    2507127
202310    2384550
202402    2367922
202401    2366130
202311    2346645
Name: SAFRA, dtype: int64

In [12]:
df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] \
    .value_counts(normalize=True) \
    .mul(100) \
    .round(2) \
    .rename('percentual_total')


PREPG    72.27
AUTOC    25.73
FLEXD     1.10
CTLFC     0.88
POSPG     0.02
MVNOD     0.00
POSRI     0.00
POSBL     0.00
POSTL     0.00
PREBL     0.00
M2MS      0.00
Name: percentual_total, dtype: float64

In [13]:
# Substituindo os valores diferentes de PREPG E AUTOC por OUTROS
df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] = (
    df_base_recarga_selecionada['COD_PLATAFORMA_ATU']
        .where(
            df_base_recarga_selecionada['COD_PLATAFORMA_ATU'].isin(['PREPG', 'AUTOC']),
            'OUTROS'
        )
)

In [14]:
df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] \
    .value_counts(normalize=True) \
    .mul(100) \
    .round(2) \
    .rename('percentual_total')


PREPG     72.27
AUTOC     25.73
OUTROS     2.00
Name: percentual_total, dtype: float64

In [15]:
df_base_recarga_selecionada = df_base_recarga_selecionada[df_base_recarga_selecionada['COD_PLATAFORMA_ATU'] == 'PREPG']

#### Ajuste dos Tipos de Dados

In [16]:
# Ajuste dos tipos de dados para agregacao
df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'] = pd.to_numeric(df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VAL_BONUS'] = pd.to_numeric(df_base_recarga_selecionada['VAL_BONUS'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VAL_REAL'] = pd.to_numeric(df_base_recarga_selecionada['VAL_REAL'], errors='coerce').fillna(0)
df_base_recarga_selecionada['FLAG_SOS'] = pd.to_numeric(df_base_recarga_selecionada['FLAG_SOS'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VALOR_SOS'] = pd.to_numeric(df_base_recarga_selecionada['VALOR_SOS'], errors='coerce').fillna(0)

In [17]:
df_base_recarga_selecionada

,NUM_CPF,DW_NUM_NTC,DAT_INSERCAO_CREDITO,HOR_INSERCAO_CREDITO,DW_NUM_CLIENTE,COD_TECNOLOGIA_DW,COD_CANAL_AQUISICAO,COD_TIPO_CREDITO,COD_PROMOCAO,VAL_CREDITO_INSERIDO,...,DW_PLANO_TARIFACAO,DW_TIPO_RECARGA,DW_TIPO_INSERCAO,DW_FORMA_PAGAMENTO,DW_INSTITUICAO,COD_GRUPO_CARTAO,DSC_GRUPO_CARTAO_WPP,FLAG_SOS,VALOR_SOS,SAFRA
0,89UU788W7TW,712481754,2023-10-09,170410,1369129520,GSM,17598,PE,-1,20.0,...,606234,-2,-2,-2,-2,UB,Rec.Online,0,0.0,202310
5,XXTU8YUY9YN,643547119,2025-02-14,144137,1134140532,GSM,17537,PE,-1,20.0,...,541200,-2,-2,-2,14475,UB,Rec.Online,0,0.0,202502
6,8WU79XXYYWU,710658217,2023-12-07,103912,1366467561,GSM,16167,PE,-1,20.0,...,606234,-2,-2,-2,14477,UB,Rec.Online,0,0.0,202312
7,7W79XUUZWZN,755062643,2024-04-02,101009,1424276880,GSM,17537,PE,-1,25.0,...,541200,-2,-2,-2,14475,UC,Rec.Online,0,0.0,202404
8,YY7T7XTTYTW,336183951,2024-05-12,115837,694490411,GSM,16357,PE,-1,20.0,...,393401,-2,-2,-2,14211,UB,Rec.Online,0,0.0,202405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51684463,ZY8WYNWTNWN,481751768,2024-05-24,92233,905904057,GSM,16212,PE,-1,30.0,...,599100,-2,-2,-2,14003,UD,Rec.Online,0,0.0,202405
51684465,WU779UWU7WU,400328337,2023-11-11,3755,995821379,GSM,-3,PE,-1,0.0,...,599100,-2,-2,-2,-1,HZ,NaoSeAplica,0,0.0,202311
51684466,ZTZWNNXNNZZ,722090283,2024-05-29,54218,1380543191,GSM,-3,PE,-1,0.0,...,609790,2,21,-2,-1,I4,NaoSeAplica,0,0.0,202405
51684467,X8XX979W8NY,697508085,2024-05-15,181610,1204347733,GSM,313,PE,-1,20.0,...,541200,-2,-2,-2,13939,UB,Rec.Online,0,0.0,202405


#### Criação das visões agregadas por SAFRA e CPF

In [18]:
df_base_recarga_selecionada_agg = df_base_recarga_selecionada.groupby(['NUM_CPF', 'SAFRA']).agg(
        QTDE_RECARGAS=('NUM_CPF', 'size'),
        QTDE_NUMEROS=('DW_NUM_NTC', 'nunique'),
        VAL_CREDITO_INSERIDO=('VAL_CREDITO_INSERIDO', 'sum'),
        VAL_BONUS=('VAL_BONUS', 'sum'),
        VAL_REAL=('VAL_REAL', 'sum'),
        QTD_SOS=('FLAG_SOS', 'sum'),
        VALOR_SOS=('VALOR_SOS', 'sum'),
        
    ).reset_index()

In [19]:
df_base_recarga_selecionada_agg

,NUM_CPF,SAFRA,QTDE_RECARGAS,QTDE_NUMEROS,VAL_CREDITO_INSERIDO,VAL_BONUS,VAL_REAL,QTD_SOS,VALOR_SOS
0,777777UWTYZ,202310,2,1,30.0,84300.0,84330.0,0,0.0
1,777777UWTYZ,202311,5,2,50.0,164901.0,164951.0,0,0.0
2,777777UWTYZ,202401,2,1,20.0,80601.0,80621.0,0,0.0
3,777777UWTYZ,202402,4,2,50.0,164901.0,164951.0,0,0.0
4,777777UWTYZ,202403,2,1,30.0,84300.0,84330.0,0,0.0
...,...,...,...,...,...,...,...,...,...
11662985,ZZZZZZZZY7Y,202411,46,23,560.0,498788.0,499348.0,1,5.0
11662986,ZZZZZZZZY7Y,202412,55,24,690.0,316764.8,317454.8,1,5.0
11662987,ZZZZZZZZY7Y,202501,51,24,605.0,257976.0,258581.0,3,25.0
11662988,ZZZZZZZZY7Y,202502,41,18,430.0,173697.0,174127.0,2,10.0


## Construção do Book Comportamental por SAFRA (Modelo de Crédito)

Este processo tem como objetivo a construção de um **book comportamental temporal** para modelagem de crédito, alinhado à predição de **FPD (First Payment Default)**.

### Visão Geral
Os dados comportamentais são inicialmente agregados no nível **CPF + SAFRA**, representando o comportamento observado em cada período mensal. A partir dessa base agregada, são criadas features históricas que capturam o comportamento passado do cliente em relação à **safra de referência**.

A base final do modelo é obtida por meio de um **LEFT JOIN** entre:
- **Base alvo (label)**: CPF + SAFRA com indicador FPD (0/1)
- **Base comportamental enriquecida**: histórico anterior ao mês da SAFRA

### Construção das Features Temporais
Para cada CPF, os dados são ordenados cronologicamente por SAFRA e são criadas defasagens temporais (*lags*) utilizando apenas informações do passado:

- Safra imediatamente anterior (t-1)
- Acumulado das últimas 3 safras (t-1 a t-3)
- Acumulado das últimas 6 safras (t-1 a t-6)

As defasagens são geradas via `groupby(CPF)` com `shift`, garantindo que **nenhuma informação da própria safra ou futura seja utilizada**, evitando vazamento temporal (*data leakage*).

### Robustez e Tratamento de Casos Especiais
- CPFs sem histórico anterior permanecem na base (LEFT JOIN), com valores nulos ou zerados.
- A ausência de histórico é considerada informação relevante para o modelo.
- O método é robusto a safras faltantes (meses sem registro), utilizando apenas o histórico efetivamente disponível.

### Garantias do Processo
- As features refletem exclusivamente o comportamento conhecido **até o momento da decisão de crédito**.
- A estrutura é reproduzível, auditável e adequada para uso em modelos supervisionados.
- O book final está preparado para técnicas de modelagem estatística e de machine learning.

Este desenho segue práticas consolidadas de modelagem de risco de crédito e permite expansão futura com métricas adicionais (médias, tendências, taxas e flags de histórico).


In [20]:
variaveis = [
    'QTDE_RECARGAS',
    'QTDE_NUMEROS',
    'VAL_CREDITO_INSERIDO',
    'VAL_BONUS',
    'VAL_REAL',
    'QTD_SOS',
    'VALOR_SOS'
]

df_book_04 = funcoes.criar_lags_por_safra(
    df=df_base_recarga_selecionada_agg,
    col_cpf='NUM_CPF',
    col_safra='SAFRA',
    variaveis=variaveis,
    janelas=[1, 3, 6]
)

In [21]:
df_book_04

,NUM_CPF,SAFRA,QTDE_RECARGAS,QTDE_NUMEROS,VAL_CREDITO_INSERIDO,VAL_BONUS,VAL_REAL,QTD_SOS,VALOR_SOS,QTDE_RECARGAS_ULT_1_SAFRAS,...,VAL_BONUS_ULT_6_SAFRAS,VAL_REAL_ULT_1_SAFRAS,VAL_REAL_ULT_3_SAFRAS,VAL_REAL_ULT_6_SAFRAS,QTD_SOS_ULT_1_SAFRAS,QTD_SOS_ULT_3_SAFRAS,QTD_SOS_ULT_6_SAFRAS,VALOR_SOS_ULT_1_SAFRAS,VALOR_SOS_ULT_3_SAFRAS,VALOR_SOS_ULT_6_SAFRAS
0,777777UWTYZ,202310,2,1,30.0,84300.0,84330.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,777777UWTYZ,202311,5,2,50.0,164901.0,164951.0,0,0.0,2.0,...,84300.0,84330.0,84330.0,84330.0,0.0,0.0,0.0,0.0,0.0,0.0
2,777777UWTYZ,202401,2,1,20.0,80601.0,80621.0,0,0.0,5.0,...,249201.0,164951.0,249281.0,249281.0,0.0,0.0,0.0,0.0,0.0,0.0
3,777777UWTYZ,202402,4,2,50.0,164901.0,164951.0,0,0.0,2.0,...,329802.0,80621.0,329902.0,329902.0,0.0,0.0,0.0,0.0,0.0,0.0
4,777777UWTYZ,202403,2,1,30.0,84300.0,84330.0,0,0.0,4.0,...,494703.0,164951.0,410523.0,494853.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11662985,ZZZZZZZZY7Y,202411,46,23,560.0,498788.0,499348.0,1,5.0,60.0,...,4437416.8,647175.0,1882073.9,4441525.8,3.0,7.0,16.0,15.0,43.0,103.0
11662986,ZZZZZZZZY7Y,202412,55,24,690.0,316764.8,317454.8,1,5.0,46.0,...,4285966.3,499348.0,1544226.4,4289874.3,1.0,7.0,11.0,5.0,43.0,63.0
11662987,ZZZZZZZZY7Y,202501,51,24,605.0,257976.0,258581.0,3,25.0,55.0,...,3689671.2,317454.8,1463977.8,3693629.2,1.0,5.0,10.0,5.0,25.0,58.0
11662988,ZZZZZZZZY7Y,202502,41,18,430.0,173697.0,174127.0,2,10.0,51.0,...,2953649.7,258581.0,1075383.8,2957457.7,3.0,5.0,12.0,25.0,35.0,78.0


In [22]:
book_04 = df_book_04.copy()

In [23]:
book_04 = pd.merge(
    book_03,
    df_book_04,
    on=['NUM_CPF', 'SAFRA'],
    how='left'
)

In [24]:
book_04

,SAFRA,FPD,SCORE_01,SCORE_02,NUM_CPF,SCORE_RATEO,SCORE_AVG,SCORE_DIFF,SCORE_MIN,DATADENASCIMENTO,...,VAL_BONUS_ULT_6_SAFRAS,VAL_REAL_ULT_1_SAFRAS,VAL_REAL_ULT_3_SAFRAS,VAL_REAL_ULT_6_SAFRAS,QTD_SOS_ULT_1_SAFRAS,QTD_SOS_ULT_3_SAFRAS,QTD_SOS_ULT_6_SAFRAS,VALOR_SOS_ULT_1_SAFRAS,VALOR_SOS_ULT_3_SAFRAS,VALOR_SOS_ULT_6_SAFRAS
0,202410,0,562.0,636.0,ZZZZZX7XWY8,1.131673,599.0,74.0,562.0,1983-12-26,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,202410,1,546.0,518.0,ZZZZZX88YXY,0.948718,532.0,-28.0,518.0,1980-12-24,...,84420.0,40.0,122.0,84572.0,0.0,1.0,1.0,0.0,5.0,5.0
2,202410,0,621.0,750.0,ZZZZZYT7XYT,1.207729,685.5,129.0,621.0,1982-07-23,...,0.0,25.0,25.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0
3,202410,1,609.0,679.0,ZZZZZNTXY9Z,1.114943,644.0,70.0,609.0,1988-09-27,...,379354.1,42241.1,126634.1,379609.1,0.0,1.0,1.0,0.0,20.0,20.0
4,202410,0,621.0,722.0,ZZZZZ79ZXUX,1.162641,671.5,101.0,621.0,1984-08-04,...,42200.0,42220.0,42240.0,42240.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1280823,202503,0,604.0,674.0,99997YWXNZZ,1.115894,639.0,70.0,604.0,1960-01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1280824,202503,0,688.0,765.0,99998TYXZN8,1.111919,726.5,77.0,688.0,1982-12-25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1280825,202503,0,616.0,630.0,9999888YYU9,1.022727,623.0,14.0,616.0,1988-12-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1280826,202503,0,627.0,649.0,9999889ZN9X,1.035088,638.0,22.0,627.0,1963-09-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
book_04['REGIAO_POSTAL_TXT'].value_counts()

Nordeste - Norte            186867
Centro-Oeste e Norte        163603
SP - Interior               150548
SP - Capital e Grande SP    145821
RJ e ES                     140769
Nordeste - Leste            120446
Nordeste - Bahia/Sergipe    108227
MG                           81622
Desconhecido                 74605
Sul - RS                     54266
Sul - PR/SC                  54054
Name: REGIAO_POSTAL_TXT, dtype: int64

In [26]:
# Sanity check
book_03.shape[0] == df_book_04.shape[0]

False

In [27]:
book_04.to_parquet(project_root/'database/processed/book_variaveis_04.parquet', index=False)